# Importação de bibliotecas 📚

In [83]:
from langchain_community.document_loaders import DirectoryLoader, BSHTMLLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter
from sys import exit
from textwrap import dedent
import torch
import re
import pickle
import uuid

# Sessão - Banco de Dados 🏦

## Objetivo Da Sessão 🧬:
* Criar um Banco de Dados Vetorial.

# Definindo a primeira classe do RAG 1️⃣.

## Principais objetivos 📝:
* Verificar se o usuário vai criar um novo Banco de Dados, se já criou, não será possível criar outro;
* Definir o modelo de Embedding;
* Carregar o Banco de Dados com Chroma;
* Verificar se o Banco de Dados está vazio;
* Pesquisar elementos do Banco de Dados.

In [84]:
class DBConnection:
    """The first class of Compass Rag Project"""

    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._inicializate = False

        return cls._instance

    @property
    def embedding_model(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-base-en-v1.5",
            model_kwargs={'device': device}
        )

    def __init__(self, directory) -> None:
        if not self._instance._inicializate: 
            try:
                self.db = Chroma (
                    persist_directory = directory,
                    embedding_function = self.embedding_model,
                    collection_metadata={"hnsw:space": "cosine"}
                )
    
                self._instance._inicializate = True

                print("✅ \033[0;32mData Bank Sucessfully created!\033[m")

            except Exception as error:
                print(f"\033[0;31m🚨 Error: {error}\033[m")
                exit(1)

        else:
            print("⚠️ \033[1;31mAlready created a Data Bank.\033[m")
            return

    def db_is_empty(self) -> bool:
        return len(self.db.get()["ids"]) == 0
        

    def search_information(self, question: str, filter: str = None, k=10) -> list:
        if not self.db_is_empty():
            available_titles = self.db.get().get("metadatas", [])
            unique_titles = list(set([m["title"] for m in available_titles if "title" in m]))

            instructional_query = f"Represent this sentence for searching relevant passages: {question}"
    
            search_kwargs = {
                "query": question,
                "k": k,
                "fetch_k": 20,
                "lambda_mult": 0.8
            }
    
            for title in unique_titles:

                if title.lower() in question.lower() or title.split(';')[0].lower() in question.lower():
                    search_kwargs["where"] = {"title": title}
                    print(f"🤖 \033[0;34mAuto-Filter applied for:\033[m {title}")
                    break
    
            results = self.db.max_marginal_relevance_search_with_score(**search_kwargs)
            return results

        else:
            return "⚠️ \033[1;31mData bank is empty! It is not possible to search any information.\033[m"

# Definindo a segunda classe do RAG 2️⃣.

## Principais objetivos 📝:
* Carregar os arquivos em uma DataBase;
* Definir os chucks e overlaps;
* Filtragem de dados com o Regex;
* Adicionar esses novos dados ao Banco de Dados existente e vetorizar-lós;

In [85]:
class VectorizingData:
    """The second class of the Compass Rag Project"""

    def __init__(self, persistent_directory: str, db: DBConnection) -> None:
        """Main constructor of the class
        
        :param persistent_directory -> The directory to be given
        :param db -> The Data Bank Created
        """
        
        self.persistent_directory = persistent_directory
        self.db = db
        self._chunks = []


    def _clean_data(self, docs: list) -> list:
        """This method have the function of clear the data with regex

        :param docs -> The brute docs to be given
        
        :returns -> The clean data
        """

        clean_docs: list = []

        for doc in docs:
            brute_docs = doc.page_content

            title_match = re.search(r"(?<=Title:)(?P<title>.+)", brute_docs)

            if title_match:
                doc.metadata["title"] = title_match.group("title").strip()

            author_match = re.search(r"(?<=Author:)(?P<author>.+)", brute_docs)

            if author_match:
                doc.metadata["author"] = author_match.group("author").strip()

            data_match = re.search(r"(?<=Release\sdate:)(?P<data>.+?)(?=\s\[)", brute_docs)

            if data_match:
                doc.metadata["data"] = data_match.group("data").strip()

            language_match = re.search(r"(?<=Language:)(?P<language>.+)", brute_docs)

            if language_match:
                doc.metadata["language"] = language_match.group("language").strip()

            
            cleaned_text = re.sub(
                r"^.*?\*\*\*\s*START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*",
                "",
                brute_docs,
                flags=re.DOTALL | re.IGNORECASE
            )


            cleaned_text = re.sub(
                r"THERE IS AN ILLUSTRATED EDITION.*?[\]]\s*", 
                "", 
                cleaned_text, 
                flags=re.IGNORECASE | re.DOTALL
            )

            doc.page_content = cleaned_text.strip()

            clean_docs.append(doc)

        return clean_docs
            

        
    def _loading_data(self) -> list | str:
        """This method have the function of loading the documents
        
        :returns -> The documents loaded
        """

        try:
            loader = DirectoryLoader(
                path = self.persistent_directory,
                glob = "*.html",
                loader_cls = BSHTMLLoader
            )
    
            brute_docs: list = loader.load()
            clean_docs = self._clean_data(brute_docs)

            return clean_docs

        except Exception as error:
            return f"\033[0;31m🚨 Error: {error}\033[m"


    def _data_chunk_and_overlap(self) -> list | str:
        """This method have the function of divide the content on chunks and do overlap between the chunks
        
        :returns -> The list of chunks getted
        """

        try:
            clean_docs = self._loading_data()
    
            chunk_splitter = RecursiveCharacterTextSplitter(
                chunk_size = 1200,
                chunk_overlap = 400
            )
    
            self._chunks = chunk_splitter.split_documents(clean_docs)

            for chunk in self._chunks:
                title = chunk.metadata.get('title', 'Unknown Book')
                
                chunk.page_content = f"Book Title: {title} | Chapter context: {chunk.page_content}"

                chunk.metadata["id"] = str(uuid.uuid4())

            return self._chunks
        
        except Exception as error:
            return f"\033[0;31m🚨 Error: {error}\033[m"


    @property
    def chunk_data_information(self) -> dict:
        """This method have the function of show some informations about the chunks
        
        :returns -> The information collected about the chunks
        """

        if len(self._chunks) == 0:
            return "⚠️ \033[1;31mYou don't have processed any chunks. Execute the method '_data_chunk_and_overlap' to work!\033[m"

        else:
            stats: dict = {
                "total_chunks": len(self._chunks),
                "per_title": {},
                "per_author": {},
                "per_language": {},
                "per_data": {},
            }

            for chunk in self._chunks:
                t = chunk.metadata.get('title', '?')
                a = chunk.metadata.get('author', '?')
                l = chunk.metadata.get('language', '?')
                d = chunk.metadata.get('data', '?')

                stats["per_title"][t] = stats["per_title"].get(t, 0) + 1
                stats["per_author"][a] = stats["per_author"].get(a, 0) + 1
                stats["per_language"][l] = stats["per_language"].get(l, 0) + 1
                stats["per_data"][d] = stats["per_data"].get(d, 0) + 1

            return stats


    def vectorizing_data(self):
        """Adds documents to ChromaDB using safe batching"""
        
        batch_size = 1000
        total = len(self._chunks)
        
        try:
            for i in range(0, total, batch_size):
                batch = self._chunks[i : i + batch_size]
                self.db.db.add_documents(batch)
                print(f"✅ Progress: {min(i + batch_size, total)}/{total} chunks vectorized.")
            return "🏆 \033[0;32mVectorization Complete!\033[m"
            
        except Exception as e:
            return f"\033[0;31m🚨 Batching Error: {e}\033[m"


    def save_to_pickle(self, filename="processed_chunks.pkl"):
        """PDF Requirement: Saves the processed data structure into a Pickle file"""
        
        serialized_data = []
        
        for chunk in self._chunks:
            serialized_data.append({
                "id": chunk.metadata.get("id"),
                "content": chunk.page_content,
                "metadata": chunk.metadata
            })
            
        with open(filename, "wb") as f:
            pickle.dump(serialized_data, f)
        print(f"📦 \033[0;33mData successfully serialized to {filename}\033[m")

# Sessão de Testes 📔

## Objetivos da sessão 🧬:
* Mostrar na prática o Embedding Trabalhando;
* Documentação de toda a minha trajetória para a realização do Desafio.

## Definindo os caminhos das pastas e das classes
- Banco de Dados
- Base de Dados

In [86]:
DATA_PATH = "data_base"
DB_PATH = "data_bank"

db_instance = DBConnection(DB_PATH)
vectorizing = VectorizingData(DATA_PATH, db_instance)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6305.84it/s]


✅ Data Bank Sucessfully created!


## Vetorizando o Banco de Dados 🏦
- Colocando os chunks em arquivos .pkl
- Aplicando os chunks e overlaps nos documentos HTML.
- Aplicando os chunks diretamente no Banco de Dados. Por conseguinte, vetorizando esses chunks.

### Processo Lógico 🧠
- O método "_data_chunk_and_overlap" vai chamar a função "_loading_data" e vai pegar os dados que já foram limpos e vai amarzena-lós para serem divididos em uma quantidade considerável de chunks e ligam os mesmos com overlaps. Contudo definimos um novo padrão para o conteúdo. Logo, todo chunk será acompanhado com o metadado "title". Outrossim, chamo o método "vectorizing_data" para guardar esses chunks dentro do banco de dados vetorizado e com a medida de segurança de 1000 batchs por vez para não estourar. Por fim, congelamos tudo e colocamos tudo em um arquivo .pkl.

In [ ]:
vectorizing._data_chunk_and_overlap()
vectorizing.vectorizing_data()
vectorizing.save_to_pickle()

## Testando o Modelo
- Fazer perguntas fáceis;
- Fazer perguntas difícies;
- Visualização dos 10 chunks para cada pergunta;
- Visualização do score dos chunks;
- Visualização da porcentagem de confiabilidade dos chunks.

In [ ]:
def main():
    """Main executor of program"""
    while True:
        try:
            prompt = "Enter with your question (or 'exit' to quit):"
            question = str(input(prompt)).strip()
    
            if not question:
                print("\033[0;31m❌ Error! Your question can't be empty.\033[m")
                continue
            
            if question.lower() in ['exit', 'quit', 'sair']:
                break

            results = db_instance.search_information(question)

            if results:
                print("\n\033[1;36m📚 The best excerpts founded:\033[0m")

                for count, (doc, distance) in enumerate(results, 1):
                    score_val = 1 - distance
                    reliability = score_val * 100
                    
                    clean_lines = [linha.strip() for linha in doc.page_content.split('\n')]
                    final_content = "\n".join(clean_lines)
                
                    output = f"""
\033[1;33mDocument {count}\033[m
\033[1;34m📊 Metrics\033[m: [Score: {distance:.4f}] | [Reliability: {reliability:.2f}%]

\033[0;31mID\033[m: {doc.metadata.get("id", "undefined")}
\033[1;31mBook's title\033[m: {doc.metadata.get("title", "undefined")}
\033[1;32mAuthor's book\033[m: {doc.metadata.get("author", "undefined")}

==================== Content =======================

{final_content}
"""
                    
                    print(output)
            else:
                print("\n\033[1;31m⚠️ No excerpts founded!\033[m")

        except KeyboardInterrupt:
            print("\n\033[0;31m❌ Process interrupted by user.\033[m")
            break
        except Exception as error:
            print(f"\033[0;31m❌ Error! {error}\033[m")
            break

In [ ]:
main()